# Stage 2: Shrink-Wrapping (Downsampled)

Shrink hull masks to minimize background while maintaining containment.

**Input**: `labelsHull_ds/*_hull.nii.gz` + `labels_ds/*.nii.gz`  
**Output**: `labelsShrunk_50_ds/*_shrunk.nii.gz`, `labelsShrunk_95_ds/*_shrunk.nii.gz`  
**Metrics**: `metrics/shrink_metrics_ds.json`

In [1]:
# Configuration - copy from 00_config.ipynb or run that notebook first
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# Shrink-wrap percentiles to generate
PERCENTILES = [50, 95]

In [2]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    load_nifti, save_nifti, save_metrics, ensure_dir,
    shrink_wrap_distance, compute_shrink_metrics
)

In [3]:
# Setup directories (using downsampled data)
target = Path(TARGET_DIR)
LABELS_DIR = target / "labels_ds"
HULL_DIR = target / "labelsHull_ds"
METRICS_DIR = ensure_dir(target / "metrics")

# Create output directories for each percentile (downsampled)
OUTPUT_DIRS = {}
for p in PERCENTILES:
    OUTPUT_DIRS[p] = ensure_dir(target / f"labelsShrunk_{p}_ds")

print(f"Original labels: {LABELS_DIR}")
print(f"Hull masks: {HULL_DIR}")
print(f"Output directories:")
for p, d in OUTPUT_DIRS.items():
    print(f"  {p}th percentile: {d}")

Original labels: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labels_ds
Hull masks: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsHull_ds
Output directories:
  50th percentile: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsShrunk_50_ds
  95th percentile: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsShrunk_95_ds


In [4]:
def process_shrink(original_path, hull_path, output_path, percentile):
    """Process a single sample with shrink-wrapping."""
    # Load masks
    original_data, affine, header = load_nifti(original_path)
    hull_data, _, _ = load_nifti(hull_path)
    
    original_mask = (original_data > 0).astype(np.uint8)
    hull_mask = (hull_data > 0).astype(np.uint8)
    
    # Apply shrink-wrapping
    shrunk_mask = shrink_wrap_distance(hull_mask, original_mask, percentile=percentile)
    
    # Compute metrics
    metrics = compute_shrink_metrics(original_mask, hull_mask, shrunk_mask)
    metrics['percentile'] = percentile
    
    # Save output
    save_nifti(shrunk_mask, affine, header, output_path)
    
    return metrics

In [5]:
# Get hull files
hull_files = sorted(HULL_DIR.glob("*_hull.nii.gz"))
print(f"Found {len(hull_files)} hull files to process")
print("="*70)

all_metrics = {p: [] for p in PERCENTILES}

for idx, hull_file in enumerate(hull_files, 1):
    # Extract sample name
    sample_name = hull_file.stem.replace('_hull', '').replace('.nii', '')
    
    # Find corresponding original mask
    original_file = LABELS_DIR / f"{sample_name}.nii.gz"
    if not original_file.exists():
        print(f"[{idx}/{len(hull_files)}] {sample_name}: Original not found, skipping")
        continue
    
    print(f"[{idx}/{len(hull_files)}] {sample_name}")
    
    for percentile in PERCENTILES:
        output_file = OUTPUT_DIRS[percentile] / f"{sample_name}_shrunk.nii.gz"
        
        try:
            metrics = process_shrink(original_file, hull_file, output_file, percentile)
            metrics['filename'] = hull_file.name
            metrics['sample_name'] = sample_name
            all_metrics[percentile].append(metrics)
            
            reduction_pct = (1 - metrics['reduction_ratio']) * 100
            print(f"    {percentile}th: {metrics['shrunk_volume']:,} voxels "
                  f"({reduction_pct:.1f}% reduced, containment={metrics['containment_ratio']:.4f})")
        except Exception as e:
            print(f"    {percentile}th: ERROR - {e}")

Found 30 hull files to process
[1/30] Digit105


    50th: 229,238 voxels (25.8% reduced, containment=1.0000)


    95th: 293,599 voxels (5.0% reduced, containment=1.0000)
[2/30] Digit10


    50th: 310,664 voxels (35.2% reduced, containment=1.0000)


    95th: 455,868 voxels (5.0% reduced, containment=1.0000)
[3/30] Digit12


...

    95th: 251,320 voxels (5.0% reduced, containment=0.8567)
[29/30] Digit97


    50th: 279,592 voxels (33.3% reduced, containment=1.0000)


    95th: 398,439 voxels (5.0% reduced, containment=1.0000)
[30/30] Digit99


    50th: 220,140 voxels (26.5% reduced, containment=0.8891)


    95th: 284,417 voxels (5.0% reduced, containment=0.8891)


In [6]:
# Save metrics (combined for all percentiles)
combined_metrics = []
for p, metrics_list in all_metrics.items():
    combined_metrics.extend(metrics_list)

metrics_file = METRICS_DIR / "shrink_metrics_ds.json"
save_metrics(combined_metrics, metrics_file)

# Summary
print("\n" + "="*70)
print("SHRINK-WRAPPING COMPLETE (DOWNSAMPLED)")
print("="*70)

for p in PERCENTILES:
    if all_metrics[p]:
        avg_containment = np.mean([m['containment_ratio'] for m in all_metrics[p]])
        avg_reduction = np.mean([1 - m['reduction_ratio'] for m in all_metrics[p]]) * 100
        print(f"\n{p}th percentile:")
        print(f"  Processed: {len(all_metrics[p])} samples")
        print(f"  Avg containment: {avg_containment:.4f}")
        print(f"  Avg reduction: {avg_reduction:.1f}%")
        print(f"  Output: {OUTPUT_DIRS[p]}")

print(f"\nMetrics: {metrics_file}")


SHRINK-WRAPPING COMPLETE (DOWNSAMPLED)

50th percentile:
  Processed: 30 samples
  Avg containment: 0.9837
  Avg reduction: 31.9%
  Output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsShrunk_50_ds

95th percentile:
  Processed: 30 samples
  Avg containment: 0.9837
  Avg reduction: 5.0%
  Output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsShrunk_95_ds

Metrics: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/metrics/shrink_metrics_ds.json
